In [1]:
import pandas as pd

# Load datasets
demo = pd.read_csv('demographics.csv')  
labs = pd.read_csv('lab_results.csv')
dosage = pd.read_csv('dosage_logs.csv')
print("Demographics:",demo.head())
print("Lab Results:",labs.head())
print("Dosage Logs:",dosage.head())
print("Demographics:",demo.info())

Demographics:    Patient_ID   Age Sex  Baseline_BP
0         101  66.0   F        114.0
1         102  29.0   M        138.0
2         103  75.0   M        156.0
3         104  35.0   M        113.0
4         105  38.0   M        121.0
Lab Results:    Patient_ID  Cholesterol  Glucose
0         101          221    194.0
1         102          282     93.0
2         103          239    115.0
3         104          295     88.0
4         105          186    137.0
Dosage Logs:    Patient_ID Drug_Type  Dose_mg
0         101   Placebo        5
1         102    Drug_A       10
2         103   Placebo       20
3         104    Drug_A       10
4         105    Drug_A        5
<class 'pandas.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Patient_ID   550 non-null    int64  
 1   Age          545 non-null    float64
 2   Sex          550 non-null    str    
 3   Baseline_BP  541

In [ ]:
#merge the demographics and lab results datasets on Patient_ID
merged = pd.merge(demo, labs, on='Patient_ID', how='inner')  # Using inner join to keep only records with matching Patient_IDs
print("Merged Data:\n", merged.head())
print("Merged Data Info:\n", merged.info())

In [ ]:
#filtering for patients where age>18 and sistolic BP > 140
filtered = merged[(merged['Age'] > 18) & (merged['Baseline_BP'] > 140)]
print("Filtered Data:", filtered.shape[0])

In [ ]:
# Now, merge the already merged data with dosage logs
final_df = pd.merge(merged, dosage, on='Patient_ID', how='inner')
print("Final Merged Data:", final_df.info())

In [ ]:
# Check for missing values
print("Missing Values in Filtered Data:\n", merged.isnull().sum())

In [ ]:
# Create a dictionary of values to fill
fill_values = {
    'Age': merged['Age'].mean(),
    'Baseline_BP': merged['Baseline_BP'].mean(),
    'Glucose': merged['Glucose'].mean()
}

# Update the whole dataframe in one go (No ChainedAssignmentError!)
merged = merged.fillna(value=fill_values)

In [ ]:
# After handling missing values, we can check the data again
print("Data after handling missing values:\n", merged.isnull().sum())   
print("Data after handling missing values:\n", merged.info())

In [ ]:
# Group by Drug_Type and calculate the mean of Glucose
efficacy_summary = final_df.groupby('Drug_Type')['Glucose'].mean()

print("Average Glucose by Treatment Group:")
print(efficacy_summary)

In [ ]:
# Use a lambda or a custom function with .apply()
final_df['Dose_Level'] = final_df['Dose_mg'].apply(lambda x: 'High' if x > 15 else 'Standard')

# Check how many high-dose patients met your inclusion criteria
print(final_df['Dose_Level'].value_counts())

In [ ]:
# After handling missing values, we can categorize patients based on their glucose levels
def categorize_glucose(val):
    if val < 100: return 'Normal'
    elif 100 <= val < 126: return 'Prediabetic'
    else: return 'Diabetic'

merged['Glucose_Status'] = merged['Glucose'].apply(categorize_glucose)
print("Data with Glucose Status:\n", merged[['Patient_ID', 'Glucose', 'Glucose_Status']].head())

In [ ]:
#now adding some graphs to visualize the data and the results of the analysis to better understand the distribution of glucose levels across treatment groups and dose levels
import matplotlib.pyplot as plt
import seaborn as sns

# Set a professional style for medical reporting
sns.set_theme(style="whitegrid")

In [ ]:
# 1. Age Distribution Histogram
plt.figure(figsize=(10, 6))
sns.histplot(final_df['Age'], bins=12, kde=True, color='teal')
plt.title('Patient Population: Age Distribution', fontsize=15)
plt.xlabel('Age (Years)')
plt.ylabel('Number of Patients')
plt.savefig('age_distribution.png')

In [ ]:
# 2. Glucose Levels by Treatment Group (Boxplot)
plt.figure(figsize=(10, 6))
sns.boxplot(x='Drug_Type', y='Glucose', data=final_df, palette='Set2')
plt.title('Clinical Outcome: Glucose Levels by Drug Arm', fontsize=15)
plt.xlabel('Treatment Group')
plt.ylabel('Glucose (mg/dL)')
plt.savefig('glucose_comparison.png')



In [ ]:
# 3. Correlation Heatmap
plt.figure(figsize=(8, 6))
# Calculate correlation only for numeric columns
correlation_matrix = final_df[['Age', 'Baseline_BP', 'Cholesterol', 'Glucose', 'Dose_mg']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0)
plt.title('Correlation Matrix of Clinical Metrics', fontsize=15)
plt.savefig('clinical_correlations.png')

plt.show()